# Multi-Agent Workflow: Planner Agent + Executor Agent

This notebook implements a simple **multi-agent system** with two agents that
work together to accomplish a goal:

- **Planner Agent** — looks at a high-level goal and breaks it into a short,
  numbered list of concrete steps. It does *not* try to solve the goal
  itself, only plan it.
- **Executor Agent** — takes each step from the plan, one at a time, and
  actually carries it out, using the results of earlier steps as context.

A plain Python function — `run_workflow()` — acts as the **orchestrator**:
it calls the Planner once, then loops through the Executor once per step,
passing results forward and printing a final summary.

### Why split it into two agents instead of one?
A single LLM call asked to "solve this goal" tends to jump straight to an
answer without a clear, inspectable line of reasoning. Splitting the job
into **Plan → Execute** makes each step small, controllable, and easy to
watch, debug, or re-run individually — the same idea used in larger
agentic systems, just without the extra machinery.

### Orchestrator styles (for context)
| Style | What it does | Used here? |
|---|---|---|
| **Static orchestrator** | Fixed, hard-coded control flow (call A, then B, then C) | ✅ Yes — `run_workflow()` |
| **Dynamic / agentic orchestrator** | An LLM itself decides what to call next, whether to retry, when to stop | ❌ Not in this beginner version |

A static orchestrator is the right starting point for a beginner practical.
It only becomes limiting if you want the workflow to adapt on its own
(e.g. re-plan after a failed step) — a natural next upgrade once this
version is understood.


## 1. Setup

We use Google's `genai` SDK directly (no framework like LangChain), so the
whole workflow is visible in plain Python. One client is created and reused
by both agents; `MODEL` is the Gemini chat model both agents call.

> **Requirement:** a `GOOGLE_API_KEY` environment variable (or however your
> `genai.Client()` is configured to authenticate) must be set before running
> this notebook.


In [23]:
from groq import Groq

client = Groq(api_key="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")
MODEL = "openai/gpt-oss-20b"

## 2. Planner Agent

The Planner's prompt is deliberately narrow: it is told to **only** output a
numbered list of steps, and explicitly told **not** to solve the goal. This
separation of concerns is what keeps the Planner's output short, structured,
and easy for code to parse — instead of a free-form essay that happens to
contain a plan somewhere inside it.

`temperature=0.3` is used to keep the plan fairly consistent and focused
rather than creatively varied between runs.


In [24]:
def planner_agent(goal):
    prompt = f"""
You are a PLANNER AGENT. Your only job is to break the goal below into a
short, numbered list of simple, executable steps (3 to 6 steps).
Do NOT solve the goal yourself. Only output the numbered steps, nothing else.

GOAL: {goal}
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()

## 3. Parsing the Plan

The Planner returns plain text like:
```
1. Research party themes
2. Set a budget
3. ...
```
`parse_steps()` converts that text into a clean Python `list[str]`, stripping
numbering/bullet prefixes (`1.`, `2)`, `-`, etc.) so the rest of the code can
work with simple strings instead of re-parsing text every time.


In [25]:
def parse_steps(plan_text):
    """Turn the planner's numbered-list text into a clean Python list of steps."""
    steps = []
    for line in plan_text.split("\n"):
        line = line.strip()
        if not line:
            continue
        # Strip a leading "1. " / "1)" / "- " style prefix, if present
        cleaned = line.lstrip("0123456789.)- ").strip()
        if cleaned:
            steps.append(cleaned)
    return steps

## 4. Executor Agent

The Executor is called **once per step**, not once for the whole plan. Two
things make this "agentic" rather than just a for-loop of independent calls:

1. It receives `previous_results` — everything earlier steps produced — so
   later steps can build on earlier ones (e.g. use a budget decided in step
   2 while executing step 4).
2. Each call is scoped to exactly one step, which keeps outputs focused and
   makes it easy to see exactly which step produced which result.


In [26]:
def executor_agent(step, previous_results):
    context = "\n".join(
        f"Step {i+1} result: {r}" for i, r in enumerate(previous_results)
    ) or "No previous steps yet."

    prompt = f"""
You are an EXECUTOR AGENT. Carry out the single step below and give a
short, direct result. Use the context from previous steps if it's relevant.

CONTEXT SO FAR:
{context}

STEP TO EXECUTE: {step}
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()

## 5. Orchestrator — `run_workflow()`

This is the piece that actually coordinates the two agents. It is **not**
an LLM call itself — it's ordinary Python control flow, which is exactly
what a *static orchestrator* is:

1. Call the Planner once to get a plan.
2. Parse the plan into individual steps.
3. Loop through the steps, calling the Executor for each one and
   accumulating results.
4. Print a final summary pairing each step with its result.


In [27]:
def run_workflow(goal):
    print(f"\n🎯 GOAL: {goal}\n")

    # 1. PLANNER creates the plan
    print("🧠 Planner Agent is creating a plan...\n")
    plan_text = planner_agent(goal)
    steps = parse_steps(plan_text)

    print("📋 PLAN:")
    for i, step in enumerate(steps, 1):
        print(f"   {i}. {step}")

    # 2. EXECUTOR carries out each step, one by one
    print("\n⚙️  Executor Agent is working through the plan...\n")
    results = []
    for i, step in enumerate(steps, 1):
        print(f"➡️  Executing step {i}: {step}")
        result = executor_agent(step, results)
        print(f"   ✅ Result: {result}\n")
        results.append(result)

    # 3. Final summary
    print("=" * 60)
    print("🏁 FINAL SUMMARY")
    print("=" * 60)
    for i, (step, result) in enumerate(zip(steps, results), 1):
        print(f"{i}. {step}\n   -> {result}\n")

## 6. Run It

Enter any goal — e.g. `"Plan a birthday party for a 10-year-old"` or
`"Write a short blog post about healthy breakfasts"` — and watch the
Planner produce a plan, then the Executor work through it step by step.


In [28]:
user_goal = input("Enter a goal for the agents to work on: ").strip()

if user_goal:
    run_workflow(user_goal)
else:
    print("No goal entered.")


🎯 GOAL: Write a short blog post about healthy breakfasts

🧠 Planner Agent is creating a plan...

📋 PLAN:
   1. Gather quick facts and ideas about healthy breakfast foods.
   2. Draft a clear outline: intro, 3-4 key breakfast options, and a brief conclusion.
   3. Write the blog post, keeping sentences concise and engaging.
   4. Review for clarity, grammar, and flow; make any necessary edits.
   5. Publish the post on your blog platform and share it on social media.

⚙️  Executor Agent is working through the plan...

➡️  Executing step 1: Gather quick facts and ideas about healthy breakfast foods.
   ✅ Result: **Quick Facts & Ideas for Healthy Breakfasts**

| Fact | Idea |
|------|------|
| **Protein boosts satiety** – 20–30 g of protein can keep you full 3–4 hrs. | Greek yogurt parfait with berries, nuts, and a drizzle of honey. |
| **Fiber slows glucose spikes** – 5–10 g fiber per meal helps regulate blood sugar. | Overnight oats with chia seeds, sliced banana, and a sprinkle of cin

## Summary

| Component | Role | Type |
|---|---|---|
| `planner_agent()` | Breaks the goal into steps | LLM call |
| `parse_steps()` | Converts plan text to a Python list | Plain code |
| `executor_agent()` | Carries out one step, with context from earlier steps | LLM call |
| `run_workflow()` | Coordinates the above in order, collects results | Orchestrator (static) |

**Possible next steps to extend this:**
- Add error handling if a step's execution fails.
- Turn `run_workflow()` into a dynamic orchestrator: after each step, ask an
  LLM whether to continue, retry, skip, or re-plan.
- Give the Executor access to real tools (web search, calculator, file
  read/write) instead of just generating text.


## 7. Extending the Workflow

The static version above always does **Plan → Execute → Execute → ... → Summary**,
with no way to recover from a bad step and no way for the Executor to do anything
except generate text.

This section adds the three upgrades suggested at the end of the notebook:

1. **Error handling** — LLM calls and tool calls can fail (rate limits, network
   blips, bad tool input). We wrap calls in retry logic and turn failures into
   an `"[ERROR] ..."` observation instead of crashing the whole run.
2. **Real tools for the Executor** — a calculator, a web search, and file
   read/write, so the Executor can *do* things instead of only describing them.
3. **A dynamic orchestrator** — instead of a fixed `for` loop, an LLM
   (`orchestrator_agent`) looks at the current state on every turn and decides
   `PLAN` / `EXECUTE` / `REPLAN` / `DONE`, the same design used in
   `multi_agent_workflow_orchestrator_prac6.py`.

These three pieces combine into a new `run_dynamic_workflow()` that replaces
the static `run_workflow()` for anyone who wants the fuller version.


### 7.1 Error handling: `call_with_retry()`

Every LLM call in this notebook goes through one small helper. If the call
raises an exception, it retries a few times with a short delay, and if it
still fails it returns `None` instead of crashing — callers decide what to
do with a failure (skip, replan, report an error, etc.).

In [29]:
import time

def call_with_retry(fn, *args, retries=3, delay=2, **kwargs):
    """Call fn(*args, **kwargs); retry on exception; return None if all attempts fail."""
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            print(f"   ⚠️  Attempt {attempt}/{retries} failed: {e}")
            if attempt < retries:
                time.sleep(delay)
    print(f"   ❌ All {retries} attempts failed. Last error: {last_err}")
    return None


### 7.2 Real tools for the Executor

Four simple tools. Each one is defensive: bad input raises an exception with
a clear message rather than silently doing the wrong thing, so the retry /
error-handling layer above has something meaningful to report back to the
Executor.

- `calculator_tool` — evaluates a arithmetic expression safely (no `eval` on
  arbitrary code, only numbers/operators).
- `web_search_tool` — hits the DuckDuckGo Instant Answer API (no key needed)
  and returns a short summary.
- `file_write_tool` / `file_read_tool` — read/write plain text files in a
  local `workspace/` folder, so the Executor can persist real output between
  steps or for the user to inspect afterwards.

In [30]:
import os
import re
import json as _json
from urllib.parse import urlencode
from urllib.request import urlopen

WORKSPACE_DIR = "workspace"
os.makedirs(WORKSPACE_DIR, exist_ok=True)


def calculator_tool(expression):
    """Safely evaluate a basic arithmetic expression, e.g. '12 * (3 + 4)'."""
    if not re.fullmatch(r"[0-9\.\+\-\*\/\(\)\s]+", expression):
        raise ValueError(f"Unsupported characters in calculator expression: {expression!r}")
    return str(eval(expression, {"__builtins__": {}}, {}))


def web_search_tool(query):
    """Look up a query using the DuckDuckGo Instant Answer API."""
    params = urlencode({"q": query, "format": "json", "no_redirect": 1, "no_html": 1})
    with urlopen(f"https://api.duckduckgo.com/?{params}", timeout=10) as response:
        data = _json.load(response)
    summary = data.get("AbstractText") or ""
    if not summary and data.get("RelatedTopics"):
        first = data["RelatedTopics"][0]
        summary = first.get("Text", "") if isinstance(first, dict) else ""
    return summary.strip() or f"No concise summary found for: {query}"


def file_write_tool(tool_input):
    """Input format: 'filename.txt::content to write'."""
    if "::" not in tool_input:
        raise ValueError("file_write expects 'filename::content'")
    filename, content = tool_input.split("::", 1)
    filename = filename.strip()
    path = os.path.join(WORKSPACE_DIR, filename)
    with open(path, "w") as f:
        f.write(content)
    return f"Wrote {len(content)} characters to {path}"


def file_read_tool(filename):
    path = os.path.join(WORKSPACE_DIR, filename.strip())
    if not os.path.exists(path):
        raise FileNotFoundError(f"No such file: {path}")
    with open(path) as f:
        return f.read()


TOOLS = {
    "calculator": calculator_tool,
    "web_search": web_search_tool,
    "file_write": file_write_tool,
    "file_read": file_read_tool,
}

TOOL_DESCRIPTIONS = """
- calculator | <expression>        e.g. calculator | (12 + 8) * 3
- web_search | <query>             e.g. web_search | population of Japan 2025
- file_write | <filename>::<text>  e.g. file_write | notes.txt::Buy balloons and cake
- file_read  | <filename>          e.g. file_read | notes.txt
"""


def run_tool(tool_name, tool_input):
    """Execute a tool, turning any failure into an '[ERROR] ...' string
    instead of raising, so the Executor agent can see the failure and adjust."""
    tool_fn = TOOLS.get(tool_name)
    if tool_fn is None:
        return f"[ERROR] Unknown tool '{tool_name}'. Available tools: {list(TOOLS)}"
    try:
        return tool_fn(tool_input)
    except Exception as e:
        return f"[ERROR] Tool '{tool_name}' failed on input {tool_input!r}: {e}"

### 7.3 Executor v2 — ReAct-style tool use

The original `executor_agent()` could only generate text. `executor_agent_v2()`
gives it a choice on every turn: call a tool, or give a final answer. It must
reply in exactly one of two formats:

```
ACTION: <tool_name> | <tool_input>
```
or
```
RESULT: <final answer for this step>
```

The function loops (up to `max_tool_calls` times): if the model asks for an
`ACTION`, we run the tool via `run_tool()` (which never raises — see 7.2),
feed the observation back to the model, and ask again. If the model runs out
of turns without giving a `RESULT`, we fall back to its last message so the
workflow still makes progress instead of hanging.

In [31]:
def _executor_llm_call(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()


def executor_agent_v2(step, previous_results, max_tool_calls=3):
    context = "\n".join(
        f"Step {i+1} result: {r}" for i, r in enumerate(previous_results)
    ) or "No previous steps yet."

    transcript = ""  # accumulates ACTION/OBSERVATION turns for this step

    base_prompt = f"""
You are an EXECUTOR AGENT with access to tools. Carry out the single step
below. If you need a tool, respond with EXACTLY:
ACTION: <tool_name> | <tool_input>

Available tools:
{TOOL_DESCRIPTIONS}

Once you have enough information, respond with EXACTLY:
RESULT: <short, direct final result for this step>

Only ever output ONE line, either an ACTION or a RESULT. Do not explain
yourself outside of that line.

CONTEXT SO FAR:
{context}

STEP TO EXECUTE: {step}
"""

    for turn in range(1, max_tool_calls + 1):
        prompt = base_prompt + transcript
        reply = call_with_retry(_executor_llm_call, prompt)

        if reply is None:
            return f"[ERROR] Executor LLM call failed repeatedly on step: {step}"

        if reply.upper().startswith("RESULT:"):
            return reply.split(":", 1)[1].strip()

        if reply.upper().startswith("ACTION:"):
            action_body = reply.split(":", 1)[1].strip()
            if "|" not in action_body:
                transcript += f"\nACTION: {action_body}\nOBSERVATION: [ERROR] malformed action, expected 'tool | input'\n"
                continue
            tool_name, tool_input = [p.strip() for p in action_body.split("|", 1)]
            print(f"      🔧 Tool call: {tool_name} | {tool_input}")
            observation = run_tool(tool_name, tool_input)
            print(f"      📎 Observation: {observation}")
            transcript += f"\nACTION: {tool_name} | {tool_input}\nOBSERVATION: {observation}\n"
            continue

        # Model didn't follow the format — treat its raw reply as the result
        return reply

    return f"[INCOMPLETE] Ran out of tool-call turns while executing: {step}"


def planner_agent_v2(goal, results_so_far=None):
    prior = ""
    if results_so_far:
        history = "\n".join(f"- {r}" for r in results_so_far)
        prior = f"\nWork already done (take it into account, don't repeat it):\n{history}\n"

    prompt = f"""
You are a PLANNER AGENT. Break the goal below into a short, numbered list
of simple, executable steps (3 to 6 steps). Do NOT solve the goal yourself.
Only output the numbered steps, nothing else.
{prior}
GOAL: {goal}
"""

    def _call():
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
        )
        return response.choices[0].message.content.strip()

    plan_text = call_with_retry(_call)
    if plan_text is None:
        return []
    return parse_steps(plan_text)


def orchestrator_agent(goal, steps, current_index, results):
    if steps is None:
        plan_status = "No plan has been created yet."
    elif current_index >= len(steps):
        plan_status = "All planned steps have been executed."
    else:
        done = "\n".join(f"- {steps[i]}: {results[i]}" for i in range(current_index)) or "None yet."
        remaining = "\n".join(f"- {s}" for s in steps[current_index:]) or "None."
        plan_status = f"Steps completed:\n{done}\n\nSteps remaining:\n{remaining}"

    prompt = f"""
You are an ORCHESTRATOR AGENT. You control a multi-agent workflow made of
a Planner and an Executor. You do not plan or execute anything yourself —
you only decide what should happen next.

GOAL: {goal}

CURRENT STATE:
{plan_status}

Decide the next action. Reply with EXACTLY ONE of these words as the first
word of your answer, optionally followed by a short reason:
- PLAN     (no usable plan exists yet — create one)
- EXECUTE  (a plan exists and steps remain — run the next step)
- REPLAN   (the existing plan is no longer adequate for the goal, e.g. a
            step kept failing)
- DONE     (the goal is already fully satisfied)
"""

    def _call():
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return response.choices[0].message.content.strip()

    text = call_with_retry(_call)
    if not text:
        return "DONE", "orchestrator call failed repeatedly; stopping safely"

    decision = text.split()[0].upper()
    if decision not in {"PLAN", "EXECUTE", "REPLAN", "DONE"}:
        decision = "DONE"
    return decision, text

### 7.4 Dynamic Orchestrator Agent

Instead of the static `for` loop, an `orchestrator_agent()` looks at the goal
and the current progress and decides the next move: `PLAN`, `EXECUTE`,
`REPLAN`, or `DONE`. This mirrors `orchestrator_agent()` in
`multi_agent_workflow_orchestrator_prac6.py`, reusing `call_with_retry` for
resilience and falling back to `DONE` if the model ever returns something
unexpected (so the loop always terminates).

In [32]:
def planner_agent_v2(goal, results_so_far=None):
    prior = ""
    if results_so_far:
        history = "\n".join(f"- {r}" for r in results_so_far)
        prior = f"\nWork already done (take it into account, don't repeat it):\n{history}\n"

    prompt = f"""
You are a PLANNER AGENT. Break the goal below into a short, numbered list
of simple, executable steps (3 to 6 steps). Do NOT solve the goal yourself.
Only output the numbered steps, nothing else.
{prior}
GOAL: {goal}
"""

    def _call():
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
        )
        return response.choices[0].message.content.strip()

    plan_text = call_with_retry(_call)
    if plan_text is None:
        return []
    return parse_steps(plan_text)


def orchestrator_agent(goal, steps, current_index, results):
    if steps is None:
        plan_status = "No plan has been created yet."
    elif current_index >= len(steps):
        plan_status = "All planned steps have been executed."
    else:
        done = "\n".join(f"- {steps[i]}: {results[i]}" for i in range(current_index)) or "None yet."
        remaining = "\n".join(f"- {s}" for s in steps[current_index:]) or "None."
        plan_status = f"Steps completed:\n{done}\n\nSteps remaining:\n{remaining}"

    prompt = f"""
You are an ORCHESTRATOR AGENT. You control a multi-agent workflow made of
a Planner and an Executor. You do not plan or execute anything yourself —
you only decide what should happen next.

GOAL: {goal}

CURRENT STATE:
{plan_status}

Decide the next action. Reply with EXACTLY ONE of these words as the first
word of your answer, optionally followed by a short reason:
- PLAN     (no usable plan exists yet — create one)
- EXECUTE  (a plan exists and steps remain — run the next step)
- REPLAN  (the existing plan is no longer adequate for the goal, e.g. a
            step kept failing)
- DONE     (the goal is already fully satisfied)
"""

    def _call():
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return response.choices[0].message.content.strip()

    text = call_with_retry(_call)
    if not text:
        return "DONE", "orchestrator call failed repeatedly; stopping safely"

    decision = text.split()[0].upper()
    if decision not in {"PLAN", "EXECUTE", "REPLAN", "DONE"}:
        decision = "DONE"
    return decision, text

### 7.5 Dynamic control loop — `run_dynamic_workflow()`

Same shape as the static `run_workflow()`, but now:

- the Orchestrator decides each move (`PLAN` / `EXECUTE` / `REPLAN` / `DONE`)
- `executor_agent_v2` can call real tools mid-step
- a step that comes back as `"[ERROR] ..."` or `"[INCOMPLETE] ..."` is still
  recorded (so the Orchestrator can see it happened and choose `REPLAN`
  instead of the loop silently getting stuck).

In [33]:
def run_dynamic_workflow(goal, max_turns=15):
    print(f"\n🎯 GOAL: {goal}\n")

    steps = None
    current_index = 0
    results = []

    for turn in range(1, max_turns + 1):
        decision, reason = orchestrator_agent(goal, steps, current_index, results)
        print(f"🧭 Orchestrator (turn {turn}): {decision}  ({reason})")

        if decision in ("PLAN", "REPLAN"):
            steps = planner_agent_v2(goal, results)
            if decision == "PLAN":
                current_index = 0
            if not steps:
                print("   ❌ Planner failed to produce a plan. Stopping.\n")
                break
            print("📋 PLAN:")
            for i, step in enumerate(steps, 1):
                print(f"   {i}. {step}")
            print()

        elif decision == "EXECUTE":
            if not steps or current_index >= len(steps):
                print("   ⚠️  Nothing left to execute; treating as DONE.\n")
                break
            step = steps[current_index]
            print(f"➡️  Executing step {current_index + 1}: {step}")
            result = executor_agent_v2(step, results)
            print(f"   ✅ Result: {result}\n")
            results.append(result)
            current_index += 1

        elif decision == "DONE":
            print("🏁 Orchestrator says the goal is satisfied. Stopping.\n")
            break

    print("=" * 60)
    print("🏁 FINAL SUMMARY")
    print("=" * 60)
    for i, result in enumerate(results, 1):
        step_label = steps[i - 1] if steps and i - 1 < len(steps) else f"Step {i}"
        print(f"{i}. {step_label}\n   -> {result}\n")


### 7.6 Run the dynamic, tool-using workflow

Try a goal that benefits from tools, e.g. `"Work out the total cost of 3
birthday cakes at $18.50 each and save a shopping note to a file"` — you
should see the Executor call `calculator` and `file_write` mid-step.

In [34]:
user_goal_v2 = input("Enter a goal for the dynamic, tool-using workflow: ").strip()

if user_goal_v2:
    run_dynamic_workflow(user_goal_v2)
else:
    print("No goal entered.")



🎯 GOAL: Write a short blog post about healthy breakfasts

🧭 Orchestrator (turn 1): PLAN  (PLAN)
📋 PLAN:
   1. Research a few nutritious breakfast ideas and their benefits.
   2. Outline the blog post structure: intro, key points, examples, conclusion.
   3. Write a concise, engaging introduction that hooks the reader.
   4. Develop the body with clear sections on each breakfast option and its health perks.
   5. Conclude with a summary and a call‑to‑action encouraging readers to try a healthy breakfast.
   6. Edit for clarity, flow, and correct any typos before publishing.

🧭 Orchestrator (turn 2): PLAN  (PLAN)
📋 PLAN:
   1. Research quick, nutritious breakfast ideas and gather supporting facts.
   2. Outline the blog post with an intro, main points, and a conclusion.
   3. Write a draft of the post, keeping it concise and engaging.
   4. Edit for clarity, flow, and correct any errors.
   5. Publish the post on the chosen blogging platform.

🧭 Orchestrator (turn 3): DONE  (PLAN: Creat

## Summary (v2)

| Component | Role | New in v2? |
|---|---|---|
| `call_with_retry()` | Retries any LLM call and turns repeated failure into a safe `None`/fallback | ✅ Error handling |
| `calculator_tool`, `web_search_tool`, `file_write_tool`, `file_read_tool` | Real actions the Executor can take | ✅ Real tools |
| `run_tool()` | Dispatches a tool call, converts exceptions into `"[ERROR] ..."` observations | ✅ Error handling |
| `executor_agent_v2()` | ReAct-style: loops between `ACTION` (tool call) and `RESULT` (final answer) | ✅ Real tools |
| `orchestrator_agent()` | LLM decides PLAN / EXECUTE / REPLAN / DONE each turn | ✅ Dynamic orchestrator |
| `run_dynamic_workflow()` | Control loop driven entirely by the Orchestrator's decisions | ✅ Dynamic orchestrator |

The original static versions (`planner_agent`, `executor_agent`,
`run_workflow`) are left untouched above, so you can compare the static and
dynamic approaches side by side, or fall back to the simpler static version
when you don't need re-planning or tools.
